# Generation of Background Events

## Load Libraries

In [1]:
import pandas as pd
import os
import pythia8
import numpy as np
import json
import math
from skhep.math.vectors import LorentzVector, Vector3D
import copy


This function is taken from FORESEE to model the two body decay of $\pi^+$ / $K^+$ to $
\mu^+$ and $\nu_{\mu}$. It provides us with the produced particle's four momentum in lab frame


In [2]:
def twobody_decay(p0, m0, m1, m2):
    """
    Decays p0 -> p1 + p2 and returns the momenta of p1 and p2 in the lab frame.
    """
    phi = np.random.uniform(0, 2 * np.pi)
    costheta  = np.random.uniform(-1,1)
    
    zaxis = Vector3D(0, 0, 1)
    #rotaxis = zaxis.cross(p0.vector)
    # if rotaxis.mag2 == 0:  # If the vector magnitude squared is zero
    #    rotaxis = Vector3D(1, 0, 0)  # Choose an arbitrary perpendicular axis
    #rotaxis = rotaxis.unit()
    rotaxis = zaxis.cross(p0.vector).unit()
    rotangle = zaxis.angle(p0.vector)

    energy1 = (m0 * m0 + m1 * m1 - m2 * m2) / (2. * m0)
    energy2 = (m0 * m0 - m1 * m1 + m2 * m2) / (2. * m0)
    momentum1 = math.sqrt(energy1 * energy1 - m1 * m1)
    momentum2 = math.sqrt(energy2*energy2-m2*m2)

    en1 = energy1
    pz1 = momentum1 * costheta
    py1 = momentum1 * math.sqrt(1.-costheta*costheta) * np.sin(phi)
    px1 = momentum1 * math.sqrt(1.-costheta*costheta) * np.cos(phi)
    p1=LorentzVector(-px1,-py1,-pz1,en1)
    if rotangle!=0: p1=p1.rotate(rotangle,rotaxis)

    en2 = energy2
    pz2 = momentum2 * costheta
    py2 = momentum2 * math.sqrt(1.-costheta*costheta) * np.sin(phi)
    px2 = momentum2 * math.sqrt(1.-costheta*costheta) * np.cos(phi)
    p2=LorentzVector(px2,py2,pz2,en2)
    if rotangle!=0: p2=p2.rotate(rotangle,rotaxis)
    
    p0.e = math.sqrt(p0.px**2 + p0.py**2 + p0.pz**2 + m0**2)  # Correct E
    
    #boost p2 in p0 restframe
    p1_lab=p1.boost(-1.*p0.boostvector)
    p2_lab=p2.boost(-1.*p0.boostvector)
    return p1_lab, p2_lab



To provide weight to the produced particle, we define the function calculate_decay_probability().The probability of decay of pion/kaon is given by 1 - $exp(-L/c\tau\times\gamma)$. Here, L is the distance to detector, c$\tau$ is the lifetime of pion/kaon and gamma is the boost factor caculated using the formula Energy_meson/mass_meson.

The decay_pion() function is defined to use two_body_decay() function specifically for meson decay.


In [3]:
def decay_meson(px, py, pz, e, pid):
    """Simulate two-body decay of a pion (PID=211) or kaon (PID=321) into mu+ and neutrino."""
    
    # Define masses based on PID
    mass_dict = {211: 0.13957, 321: 0.49367}
    branching_fraction = {211: 0.99, 321: 0.63}
    c_tau_values = {211: 7.8, 321: 3.71}
    muon_mass = 0.10566  # GeV/c^2
    neutrino_mass = 0.0  # GeV/c^2 (assumed negligible)
    detector_distance =10    ###Distance to detector in meters 
    c_tau = c_tau_values[pid]
    particle_mass = mass_dict[pid]
    
    

    # Particle 4-momentum
    p4_particle = LorentzVector(px, py, pz, e)    
    #Decay probability calculation
    gamma = e / particle_mass  
    mean_decay_length = c_tau * gamma 
    decay_probability = (1 - np.exp(-detector_distance / mean_decay_length))*branching_fraction[pid]
    # Perform the decay
    muon, neutrino = twobody_decay(p4_particle, particle_mass, muon_mass, neutrino_mass) 
    return muon, neutrino, decay_probability


## Function that Generates Events for fixed Energy 

In the new addition part I have added a condition that if the generated particle is pion/kaon then apply the decay_meson()function to copy of the original event. 

In each copy only one meson is decayed and the coresponding  weight is calculated. The produced muon and neutrino is placed in place of the parent particle. 

The maxmimum muon energy for the event is tracked by the highest_muon_energy variable and only highest_muon_energy>30 events are saved. 

The logic is this: in the event generated by pythia, if muon+ with energy greater than 30 GeV is detected it is added to the final events_data array. The energy of mu+ is also noted to later compare it with displaced event energy. If pion/kaon is present even if muon+ is not present, the particles are marked for decay. The copies of the event with one meson decay per copy is added to the events_data array considering that the produced muon+ carries energy >30GeV and that the muon+ energy is greater than the prompt muon+ energy. In the process, the ievent is incremented by one unit for each copy so same ievent can be assigned to two events but it does not affect the calculations. It can later be fixed using the fix_dataframe() function.



In [4]:
def generate_events(energy, nevent=100000, level='hadron', target='W', process='CC_numu', felix_user=False):
    
    # Setup Pythia
    pythia = pythia8.Pythia()

    # Set random seeds
    pythia.readString("Random:setSeed=on")
    pythia.readString("Random:seed=0")

    # Define beams (frameType = 2 for fixed target collisions)
    pythia.readString("Beams:frameType = 2")
    pythia.readString("Beams:eA = "+str(energy))
    pythia.readString("Beams:eB = 0")

    # Use the nuclear PDF nCTEQ15 for Tungsten (A=184, Z=74)
    if target == 'W':
        pdffile = "nCTEQ15FullNuc_184_74_0000.dat"
    elif target == 'Fe':
        pdffile = "nCTEQ15FullNuc_56_26_0000.dat"    
    
    if felix_user: 
        pdfpath = "/Users/felixkling/work/LHAPDF-6.2.1/installed/share/LHAPDF/" + pdffile[:-9] + "/"
        pythia.readString("PDF:pSet=LHAGrid1:" + pdfpath + pdffile)
    else:
        pythia.readString("PDF:pSet=lhagrid1:" + pdffile)

    # Activate necessary options for Pythia
    pythia.readString("PDF:lepton = off")
    pythia.readString("TimeShower:QEDshowerByL = off")

    # Fix Renormalization/Factorization scale
    pythia.readString("SigmaProcess:factorScale2=6")
    pythia.readString("SigmaProcess:renormScale2=6")

    # Minimal phase space cuts
    pythia.readString("PhaseSpace:mHatMin=1")
    pythia.readString("PhaseSpace:pTHatMin=0")
    pythia.readString("PhaseSpace:pTHatMinDiverge=0")

    # Define process type and set the beam accordingly
    if process == "CC_numu_displaced":
        pythia.readString("WeakBosonExchange:ff2ff(t:W)=on")
        pythia.readString("Beams:idA = 14")
    elif process == "CC_nuebar_displaced":
        pythia.readString("WeakBosonExchange:ff2ff(t:W)=on")
        pythia.readString("Beams:idA = -12")  
    elif process == "NC_numu_displaced":
        pythia.readString("WeakBosonExchange:ff2ff(t:gmZ)=on")
        pythia.readString("Beams:idA = 14")
    elif process == "NC_nuebar_displaced":
        pythia.readString("WeakBosonExchange:ff2ff(t:gmZ)=on")
        pythia.readString("Beams:idA = -12")  
    else:
        print("Error: process_type = " + process + " is not valid.")
        return pd.DataFrame()
    
    # Decide if shower and hadronization should be included
    if level == "hard":
        pythia.readString("HadronLevel:all=off")
        pythia.readString("PartonLevel:all=off")
    elif level == "parton":
        pythia.readString("HadronLevel:all=off")
        pythia.readString("PartonLevel:all=on")
    elif level == "hadron":
        pythia.readString("HadronLevel:all=on")
        pythia.readString("PartonLevel:all=on")
    else:
        print("Error: level = " + level + " is not a valid option")
        return False

    
    # Initialize Pythia
    pythia.init()
    
    # List to store events data for DataFrame
    events_data = []
    
    # Loop over events
    for ievent in range(nevent):
        
        # Generate next event
        if not pythia.next():
            continue
        
        particles = pythia.process if level == "hard" else pythia.event
        
        # Collect all particles in the event
        event_data = []
        decay_candidates = []  # Store pion and kaon indices
        iiparticle =0
        max_muon_energy = 0
        
        for iparticle, particle in enumerate(particles):
            # Reject non-final state particles
            if particle.status() < 0:
                continue
            
            # Collect particle data
            pid = particle.id()
            px, py, pz, e = round(particle.px(), 5), round(particle.py(), 5), round(particle.pz(), 5), round(particle.e(), 5)
            parent_pid1 = particles[particle.mother1()].id()
            parent_pid2 = particles[particle.mother2()].id()
            weight = 1  # Default event weight
            
            # Store event data            
            event_data.append([ievent, iiparticle, energy, pid, px, py, pz, e, parent_pid1, parent_pid2, weight])
            iiparticle +=1
            
            # Track highest energy prompt mu+
            if pid == -13 and e > max_muon_energy:
                max_muon_energy = e
            
            # If particle is a pion+ or kaon+, mark for decay
            if pid in {211, 321} and e > 30:          
                decay_candidates.append(len(event_data) - 1)  # Store index

        
        # Store the original event data first 
        #if max_muon_energy > 30:
         #   events_data.extend(event_data)
        
        
        # Generate decayed event copies for each meson
        for i, decay_index in enumerate(decay_candidates, start=1):
            event_copy = [row[:] for row in event_data]  # Deep copy of event data
            #event_copy = event_data.copy()
            new_ievent = ievent + i  # unique ievent index for decayed event
            
            # Get particle to decay
            decay_particle = event_copy[decay_index]
            pid, px, py, pz, e = decay_particle[3], decay_particle[4], decay_particle[5], decay_particle[6], decay_particle[7]
            
            # Perform decay into muon+ and neutrino
            muon, neutrino, decay_probability = decay_meson(px, py, pz, e, pid)
                        
            
            # Check muon energy threshold
            if muon.e > 30 and muon.e > max_muon_energy:
            # Replace pion/kaon with muon+
                 decay_particle[3] = -13  # Muon+
                 decay_particle[4], decay_particle[5], decay_particle[6], decay_particle[7] = muon.px, muon.py, muon.pz, muon.e
                 decay_particle[8] = pid
                 decay_particle[9] = 90
                 decay_particle[10] = decay_probability  # Update weight

                 # Update all ievent values in this decay copy
                 event_copy = [[new_ievent] + row[1:] for row in event_copy]

                 # Add neutrino as a new particle
                 neutrino_data = [new_ievent, len(event_copy), energy, 14, neutrino.px, neutrino.py, neutrino.pz, neutrino.e, pid, 90, decay_probability]
                 event_copy.append(neutrino_data)

                 # Append the updated event to the final list
                 events_data.extend(event_copy)
    
    
    # Create output directory corresponding to different process
    output_dir = "output_events_"+process
    if not os.path.isdir(output_dir): os.makedirs(output_dir)
 
    # Convert collected event data to DataFrame
    columns = ["ievent", "iparticle", "truth_energy", "pid", "px", "py", "pz", "e", "parent_pid1", "parent_pid2","weight"]
    df = pd.DataFrame(events_data, columns=columns)  
    # See if data frame already exist, if yes, merge. 
    
    csv_file = output_dir+"/events_new"+str(energy)+".csv.zip"
    if os.path.exists(csv_file): 
        df_old = pd.read_csv(csv_file, compression="zip")
        nevt_old = df_old['ievent'].max()+1
        df['ievent'] = df['ievent']+ nevt_old
        df = pd.concat([df_old, df]) 
        
    # Export
    df.to_csv(csv_file, index=False, compression='zip')
    print("Saved events for energy "+str(energy)+" to "+csv_file)
    
    # Save number of generated events
    logging_filename = output_dir+"/generated_events.json"
    if os.path.exists(logging_filename): 
        with open(logging_filename, "r") as f: 
            logging_file = json.load(f)
    else: logging_file = {}
    if str(energy) in logging_file: logging_file[str(energy)] += nevent
    else: logging_file[str(energy)] = nevent
    with open(logging_filename, "w") as f: 
        json.dump(logging_file, f, indent=4)

## Use Function to Generate 100 Events per Energy

In [5]:
# List of energies
energies = [ 14.21, 17.90, 22.53, 28.37, 35.71, 44.964, 56.60,
     71.26, 89.71, 112.94, 142.19, 179.00, 225.35,
     283.70, 357.16, 449.64,566.07, 712.64, 897.16,
    1129.46, 1421.90
     ] 
#112.94, 142.19, 179.00, 225.35, 283.70, 357.16, 449.64,566.07, 712.64,
##14.21, 17.90, 22.53, 28.37, 35.71, 44.964, 56.60,
#     71.26, 89.71, 112.94, 142.19, 179.00, 225.35,
#     283.70, 357.16, 449.64,566.07, 712.64, 897.16,
#    1129.46, 1421.90, #1790.077754, 2253.574373, 2837.082046, 3571.674683,
    #4496.472021, 5660.722891, 7126.427896, 8971.641174
processes = [ "CC_numu_displaced", "CC_nuebar_displaced", "NC_nuebar_displaced","NC_numu_displaced"]

# Loop through all energies
for process in processes: 
    for energy in energies:
        generate_events(energy=energy, nevent=10**4, level='hadron', target='Fe', process=process, felix_user=False)


 *------------------------------------------------------------------------------------* 
 |                                                                                    | 
 |  *------------------------------------------------------------------------------*  | 
 |  |                                                                              |  | 
 |  |                                                                              |  | 
 |  |   PPP   Y   Y  TTTTT  H   H  III    A      Welcome to the Lund Monte Carlo!  |  | 
 |  |   P  P   Y Y     T    H   H   I    A A     This is PYTHIA version 8.310      |  | 
 |  |   PPP     Y      T    HHHHH   I   AAAAA    Last date of change: 25 Jul 2023  |  | 
 |  |   P       Y      T    H   H   I   A   A                                      |  | 
 |  |   P       Y      T    H   H  III  A   A    Now is 23 Feb 2025 at 00:46:51    |  | 
 |  |                                                                              |  | 
 |  |   Program docu

/var/folders/rs/w38byv0x4wlcgrjkdwx6rm_c0000gp/T/ipykernel_98176/2234408508.py:171: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df_old, df])


Saved events for energy 44.964 to output_events_CC_nuebar_displaced/events_new44.964.csv.zip

 *------------------------------------------------------------------------------------* 
 |                                                                                    | 
 |  *------------------------------------------------------------------------------*  | 
 |  |                                                                              |  | 
 |  |                                                                              |  | 
 |  |   PPP   Y   Y  TTTTT  H   H  III    A      Welcome to the Lund Monte Carlo!  |  | 
 |  |   P  P   Y Y     T    H   H   I    A A     This is PYTHIA version 8.310      |  | 
 |  |   PPP     Y      T    HHHHH   I   AAAAA    Last date of change: 25 Jul 2023  |  | 
 |  |   P       Y      T    H   H   I   A   A                                      |  | 
 |  |   P       Y      T    H   H  III  A   A    Now is 23 Feb 2025 at 00:48:29    |  | 
 |  |           

## Generate 100 more Events for E=1421.909302

In [6]:
# !python --version



In [7]:
# generate_events(energy=1421.90, nevent=1000, level='hadron', target='Fe', process='CC_numu', felix_user=True)

In [8]:
# data = pd.read_csv("output_events_NC_nuebar/events_new_bland1129.46.csv")
# #data
# pd.set_option("display.max_rows", None)  
# pd.set_option("display.max_columns", None)  
# pd.set_option("display.expand_frame_repr", False) 
# print(data)